## Install library for recsys

In [1]:
!pip3 install implicit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 56.3 MB/s eta 0:00:0000:0100:01


## Import all libraries which you will need

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import implicit # Fast Python Collaborative Filtering for Implicit Datasets.
from scipy.sparse import coo_matrix # SciPy is a scientific computation library that uses NumPy underneath.


## Load the data 

In [5]:
import pandas as pd
interactions = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/interactions.csv")
item_metadata = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/item_metadata.csv")
# user_metadata = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/user_metadata.csv")
test = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/test.csv")#['user_id'].unique()

In [6]:
train = pd.merge(interactions, item_metadata[['item_id','track_duration']], on='item_id', how='left')

In [7]:
train = train.sort_values('listened_datetime')

In [8]:
import pandas as pd
from tqdm import tqdm
import numpy as np

# train = pd.read_csv('train.csv')
# test = pd.read_csv('test.csv')

# создаем колонку listen_ratio
train['listen_ratio'] = train['listened_duration'] / train['track_duration']

# преобразуем дату в datetime, автоматически определяя формат
train['listened_datetime'] = pd.to_datetime(train['listened_datetime'], errors='coerce')

# проверяем, есть ли пропуски после преобразования
missing_dates = train['listened_datetime'].isna().sum()
print(f"Пропущенные даты после преобразования: {missing_dates}")

# создаем колонку day
train['day'] = (train['listened_datetime'] - train['listened_datetime'].min()).dt.days


# добавляем экспоненциальный коэффициент
# пример: коэффициент = exp(alpha * day), где alpha регулирует влияние свежести
alpha = 0.1
train['weight'] = np.exp(alpha * train['day'])

# умножаем listen_ratio на вес
train['weighted_listen'] = train['listen_ratio'] * train['weight']

# вычисляем глобальный топ 50
top_global = (
    train.groupby('item_id')['weighted_listen']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index
    .tolist()
)

results = []

for user_id in tqdm(test['user_id'].unique()):
    user_tracks = train[train['user_id'] == user_id]
    user_top = (
        user_tracks.sort_values('weighted_listen', ascending=False)
        .head(50)['item_id']
        .tolist()
    )
    if len(user_top) < 50:
        needed = 50 - len(user_top)
        supplement = [x for x in top_global if x not in user_top][:needed]
        user_top += supplement
    for rank, item_id in enumerate(user_top, 1):
        results.append((user_id, user_top[0], rank))

submission = pd.DataFrame(results, columns=['user_id', 'item_id', 'rank'])
submission.reset_index(inplace=True)
submission.rename(columns={'index':'id'}, inplace=True)
submission.to_csv('submission.csv', index=False)

Пропущенные даты после преобразования: 468


100%|██████████| 1500/1500 [00:34<00:00, 43.34it/s]


In [9]:
submission

,id,user_id,item_id,rank
0,0,97,39978786,1
1,1,97,39978786,2
2,2,97,39978786,3
3,3,97,39978786,4
4,4,97,39978786,5
...,...,...,...,...
74995,74995,6166700,38701558,46
74996,74996,6166700,38701558,47
74997,74997,6166700,38701558,48
74998,74998,6166700,38701558,49


In [4]:
interactions['user_id'].nunique()

314142

In [5]:
from tqdm import tqdm
tqdm.pandas()

item_metadata['track_genres_list'] = item_metadata['track_genres_list'].progress_apply(lambda x: eval(x) if isinstance(x, str) else None)
item_metadata['track_genre'] = item_metadata['track_genres_list'].progress_apply(lambda x: x[0] if isinstance(x, list) else None)
item_metadata = item_metadata.drop(columns=['track_genres_list'])

100%|██████████| 348754/348754 [00:00<00:00, 1446454.17it/s]


In [6]:
inter_df = interactions.merge(item_metadata, on='item_id', how='left')
inter_df = inter_df.merge(user_metadata, on='user_id', how='left')

In [7]:
inter_df = inter_df[~inter_df['track_duration'].isna()]

In [8]:
inter_df['listened_ratio'] = inter_df['listened_duration']/(inter_df['track_duration']+1e-6)

In [9]:
inter_df = inter_df.drop_duplicates()

In [10]:
import gc
gc.collect()

14

## Creating matrix for the model

In [11]:
inter_df = inter_df.fillna(0)

In [12]:
# Create sparse matrix
user_ids = inter_df['user_id'].astype('category')
item_ids = inter_df['item_id'].astype('category')

user_cat = user_ids.cat.codes
item_cat = item_ids.cat.codes

# Sparse matrix
interaction_matrix = coo_matrix(
    (inter_df['listened_ratio'].values, (user_cat, item_cat))
).tocsr()
import os
print(f"CPU cores: {os.cpu_count()}")

CPU cores: 4


## Training 

In [20]:
model = implicit.als.AlternatingLeastSquares(factors=4, iterations=30, num_threads=4)
model.fit(interaction_matrix)

  0%|          | 0/30 [00:00<?, ?it/s]

In [1]:
!pip install lightfm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 5.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [19]:
# --- Импорт библиотек ---
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from lightfm import LightFM
from lightfm.data import Dataset
from scipy.sparse import coo_matrix

# --- Загрузка данных ---
interactions = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/interactions.csv")
item_metadata = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/item_metadata.csv")
user_metadata = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/user_metadata.csv")
test_ids = pd.read_csv("/kaggle/input/hear-me-personalized-music-recommender/test.csv")['user_id'].unique()

In [21]:
interactions = interactions.merge(user_metadata, on='user_id', how='left')
interactions = interactions.merge(item_metadata, on='item_id', how='left')

In [22]:
interactions = interactions[~interactions['track_duration'].isna()]
interactions['target'] = interactions['listened_duration']/(interactions['track_duration']+1e-6)
interactions = interactions.drop(columns=['listened_duration'])

In [27]:
# 1. Выбираем только необходимые столбцы для Collaborative Filtering
data = interactions[['user_id', 'item_id', 'target']].copy()

# 2. Кодируем ID пользователей и песен
# LightFM требует целочисленные ID, но сначала Dataset преобразует их.
# Мы будем использовать LightFM.Dataset для простоты.
dataset = Dataset()
dataset.fit(
    users=data['user_id'].unique(),
    items=data['item_id'].unique()
)

# 3. Создаем матрицу взаимодействий (user_id, item_id, play_count)
# LightFM.Dataset.build_interactions создает (interactions, weights)
interaction_generator = (tuple(x) for x in tqdm(data[['user_id', 'item_id', 'target']].values))

# 3. Создаем матрицу взаимодействий
(interactions_matrix, weights_matrix) = dataset.build_interactions(
    interaction_generator
)

print(f"Размерность матрицы взаимодействий: {interactions_matrix.shape}")

100%|██████████| 24718189/24718189 [01:15<00:00, 329119.09it/s]


Размерность матрицы взаимодействий: (313995, 348754)


In [ ]:
model = LightFM(loss='warp', no_components=4, random_state=42)

# Обучаем модель. weights_matrix используется в качестве весов взаимодействия
# для модели WARP (Weight-Aware Rank Pairwise).
model.fit(
    interactions=interactions_matrix,
    sample_weight=weights_matrix,
    epochs=10,
    num_threads=4,
    verbose=True
)

print("\nМодель LightFM успешно обучена.")

Epoch:  20%|██        | 2/10 [00:54<03:36, 27.05s/it]

In [3]:
item_metadata.fillna('', inplace=True)
user_metadata.fillna('', inplace=True)

interactions = interactions.merge(user_metadata, on='user_id', how='left')
interactions = interactions.merge(item_metadata, on='item_id', how='left')

In [4]:
interactions = interactions.drop_duplicates()

In [5]:
interactions = interactions[~interactions['track_duration'].isna()]
interactions['listened_ratio'] = interactions['listened_duration']/(interactions['track_duration']+1e-6)

In [6]:
interactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24686266 entries, 0 to 24798185
Data columns (total 19 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   user_id                      int64  
 1   item_id                      int64  
 2   listened_duration            int64  
 3   listened_datetime            object 
 4   age_bin                      object 
 5   children                     float64
 6   gender                       object 
 7   top_genre                    object 
 8   user_disliked_track_count    float64
 9   user_liked_track_count       float64
 10  user_downloaded_track_count  float64
 11  track_name                   object 
 12  artist_name                  object 
 13  track_duration               float64
 14  track_genres_list            object 
 15  track_dislike_count          float64
 16  track_like_count             float64
 17  track_download_count         float64
 18  listened_ratio               float64
dtypes: 

In [ ]:
# --- 2. Target Encoding ---
# Преобразуем категориальные признаки
categorical_cols = interactions.select_dtypes(exclude=[np.number]).columns.tolist()
from tqdm import tqdm
# Применяем Target Encoding
for col in tqdm(categorical_cols):
    te = TargetEncoder(cols=[col])
    interactions[col] = te.fit_transform(interactions[col], interactions['listened_ratio'])

In [9]:
interactions = interactions.select_dtypes(include=[np.number])

In [11]:
unique_users = interactions['user_id'].unique()
unique_items = interactions['item_id'].unique()

user2idx = {user: idx for idx, user in enumerate(unique_users)}
item2idx = {item: idx for idx, item in enumerate(unique_items)}

interactions['user_id_enc'] = interactions['user_id'].map(user2idx)
interactions['item_id_enc'] = interactions['item_id'].map(item2idx)

In [12]:
cols = interactions.columns.tolist()
cols

['user_id',
 'item_id',
 'listened_duration',
 'children',
 'user_disliked_track_count',
 'user_liked_track_count',
 'user_downloaded_track_count',
 'track_duration',
 'track_dislike_count',
 'track_like_count',
 'track_download_count',
 'listened_ratio',
 'user_id_enc',
 'item_id_enc']

In [13]:
cols.remove('user_id')
cols.remove('item_id')
cols.remove('item_id_enc')
cols.remove('user_id_enc')
cols.remove('listened_ratio')
cols.remove('listened_duration')

In [15]:
interaction_matrix = coo_matrix(
    (interactions['listened_ratio'].astype(np.float32), 
     (interactions['user_id_enc'], interactions['item_id_enc']))
)

lfm_dataset = Dataset()
lfm_dataset.fit(users=interactions['user_id_enc'].unique(),
                items=interactions['item_id_enc'].unique(),
                user_features=cols,
                item_features=cols)

interaction_matrix = coo_matrix(
    (interactions['listened_ratio'].astype(np.float32),
     (interactions['user_id_enc'], interactions['item_id_enc']))
)

# --- 5. LightFM Dataset для фичей ---
lfm_dataset = Dataset()
lfm_dataset.fit(users=interactions['user_id_enc'].unique(),
                items=interactions['item_id_enc'].unique())

# Создаем user/item features через target encoding
user_features_list = []
item_features_list = []

In [18]:
# 1. Select the relevant columns (user_id and feature columns)
user_cols = ['user_id_enc'] + cols

# 2. Drop duplicates on user_id_enc to get one representative row per user.
# The 'first' one is used, which is equivalent to .values[0] in your original code.
unique_user_features_df = interactions[user_cols].drop_duplicates(subset=['user_id_enc'], keep='first')

# 3. Define a function to format the features for a single row
def format_features(row):
    feats = []
    for col in cols:
        # Use .get(col) for robustness, although it should exist if unique_user_features_df is correctly built
        feats.append(f"{col}_{row[col]:.4f}")
    return feats

# 4. Apply the function to each row to create the feature list
# axis=1 applies the function across rows
formatted_features = unique_user_features_df.apply(format_features, axis=1)

# 5. Create the final list of (user_id, features_list) tuples
user_features_list_fast = list(zip(unique_user_features_df['user_id_enc'], formatted_features))

# Now you can use the faster list:
user_features = lfm_dataset.build_user_features(user_features_list_fast)

# 1. Select the relevant columns (item_id and feature columns)
item_cols_to_keep = ['item_id_enc'] + [c for c in cols if c in interactions.columns]

# 2. Drop duplicates on item_id_enc to get one representative row per item.
unique_item_features_df = interactions[item_cols_to_keep].drop_duplicates(subset=['item_id_enc'], keep='first')

# 3. Define a function to format the features for a single row
def format_item_features(row):
    feats = []
    for col in cols:
        # Check if the column exists in the row (DataFrame)
        if col in item_cols_to_keep:
             feats.append(f"{col}_{row[col]:.4f}")
    return feats

# 4. Apply the function to each row to create the feature list
# axis=1 applies the function across rows
formatted_item_features = unique_item_features_df.apply(format_item_features, axis=1)

# 5. Create the final list of (item_id, features_list) tuples
item_features_list_fast = list(zip(unique_item_features_df['item_id_enc'], formatted_item_features))

# Now you can use the faster list:
item_features = lfm_dataset.build_item_features(item_features_list_fast)

# --- 6. Обучение LightFM ---
model = LightFM(loss='warp', no_components=32, learning_rate=0.05, random_state=42)
model.fit(interaction_matrix, user_features=user_features, item_features=item_features, epochs=10, num_threads=4)

# --- 7. Предсказание для тестовых пользователей ---
test_user_enc = user_encoder.transform(test_ids)

predictions = {}
for u_enc in test_user_enc:
    scores = model.predict(u_enc, np.arange(len(item_encoder.classes_)), user_features=user_features, item_features=item_features)
    top_items = item_encoder.inverse_transform(np.argsort(-scores)[:10])  # Топ-10
    predictions[user_encoder.inverse_transform([u_enc])[0]] = top_items

# Пример вывода для одного пользователя
print(predictions[test_ids[0]])

ValueError: Feature children_0.0000 not in feature mapping. Call fit first.

## Generate recommendations


In [21]:
from tqdm import tqdm
tqdm.pandas()

user_id_to_cat = dict(zip(user_ids, user_cat))
item_cat_to_id = dict(zip(item_cat, item_ids))

TypeError: 'numpy.int32' object is not iterable

In [22]:
recommendations = []
for user_id in tqdm(test_ids):
    if user_id in user_id_to_cat:
        u_cat = user_id_to_cat[user_id]
        item_cats, scores = model.recommend(u_cat, interaction_matrix[u_cat], N=50)
        
        for rank, (item_cat, score) in enumerate(zip(item_cats, scores), 1):
            item_id = item_cat_to_id[item_cat]
            recommendations.append({'user_id': user_id, 'item_id': item_id, 'rank': rank})
submission  = pd.DataFrame(recommendations)
submission.insert(0, 'id', range(len(submission)))

100%|██████████| 1500/1500 [00:02<00:00, 576.27it/s]


# Catboost Ranker

In [9]:
inter_df = (
    inter_df
    .groupby('user_id', group_keys=False)
    .head(100)
    .reset_index(drop=True)
)

In [10]:
import pandas as pd
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import train_test_split
import numpy as np

cat_features = inter_df.select_dtypes(exclude=[np.number]).columns.tolist()
cat_features.remove('listened_datetime')
inter_df[cat_features] = inter_df[cat_features].fillna('')

target = inter_df['listened_ratio']

feature_cols = [col for col in inter_df.columns if col not in ['listened_ratio', 'listened_duration', 'listened_datetime']]
X = inter_df[feature_cols]

unique_users = inter_df['user_id'].unique()
train_users, test_users = train_test_split(unique_users, test_size=0.2, random_state=42)

train_df = inter_df[inter_df['user_id'].isin(train_users)]
test_df = inter_df[inter_df['user_id'].isin(test_users)]

In [11]:
def make_group_sizes(df):
    return df.groupby('user_id').size().values

X_train = train_df[feature_cols]
y_train = train_df['listened_ratio']
group_train = make_group_sizes(train_df)

X_test = test_df[feature_cols]
y_test = test_df['listened_ratio']
group_test = make_group_sizes(test_df)

In [12]:
del train_df, X
gc.collect()

0

In [13]:
ranker = CatBoostRanker(
    iterations=500,
    learning_rate=0.05,
    depth=4,
    loss_function='YetiRank',
    eval_metric='NDCG',
    random_seed=42,
    task_type='GPU',
    verbose=50
)

train_pool = Pool(X_train, label=y_train, group_id=X_train['user_id'], group_weight=None, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features, group_id=X_test['user_id'])

ranker.fit(train_pool, eval_set=test_pool, use_best_model=True)

Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8194856	best: 0.8194856 (0)	total: 705ms	remaining: 5m 51s
50:	test: 0.8585697	best: 0.8585697 (50)	total: 24.1s	remaining: 3m 32s
100:	test: 0.8645857	best: 0.8645857 (100)	total: 47.7s	remaining: 3m 8s
150:	test: 0.8657577	best: 0.8657577 (150)	total: 1m 11s	remaining: 2m 45s
200:	test: 0.8661531	best: 0.8661531 (200)	total: 1m 35s	remaining: 2m 21s
250:	test: 0.8663594	best: 0.8663821 (245)	total: 1m 59s	remaining: 1m 58s
300:	test: 0.8666675	best: 0.8666679 (295)	total: 2m 23s	remaining: 1m 34s
350:	test: 0.8668583	best: 0.8668745 (340)	total: 2m 47s	remaining: 1m 11s
400:	test: 0.8669078	best: 0.8669672 (375)	total: 3m 11s	remaining: 47.4s
450:	test: 0.8671104	best: 0.8671104 (450)	total: 3m 36s	remaining: 23.5s
499:	test: 0.8671548	best: 0.8671548 (499)	total: 3m 59s	remaining: 0us
bestTest = 0.8671547673
bestIteration = 499


In [14]:
from tqdm import tqdm
import pandas as pd
import numpy as np

tqdm.pandas()

recommendations = []

# Список всех айтемов
all_items = inter_df['item_id'].unique()

# Цикл по пользователям
for user_id in tqdm(test_ids):
    # Проверяем, что юзер известен
    if user_id not in user_metadata['user_id'].values:
        continue

    # --- Формируем DataFrame кандидатов ---
    candidates = pd.DataFrame({'item_id': all_items})
    candidates['user_id'] = user_id  # добавляем юзера

    # --- Мержим фичи ---
    # Фичи айтемов
    candidates = candidates.merge(item_metadata, on='item_id', how='left')
    # Фичи юзера (дублируем для всех айтемов кандидатов)
    user_feats = user_metadata[user_metadata['user_id'] == user_id]
    for col in user_feats.columns:
        if col != 'user_id':
            candidates[col] = user_feats.iloc[0][col]

    # --- Оставляем только признаки, которые были на обучении ---
    feature_cols = X_train.columns.tolist()
    candidates[cat_features] = candidates[cat_features].fillna('')
    candidates_pool = Pool(candidates[feature_cols], cat_features=cat_features)

    # --- Предсказание ---
    scores = ranker.predict(candidates_pool)

    # --- Сохраняем топ-N ---
    top_N = 50
    top_idx = np.argsort(scores)[::-1][:top_N]

    top_items = candidates.iloc[top_idx].copy()
    top_items['score'] = scores[top_idx]
    top_items['rank'] = np.arange(1, len(top_items) + 1)

    recommendations.extend(top_items[['user_id', 'item_id', 'rank']].to_dict('records'))

# --- Финальный DataFrame для сабмишна ---
submission = pd.DataFrame(recommendations)
submission.insert(0, 'id', range(len(submission)))

100%|██████████| 1500/1500 [27:19<00:00,  1.09s/it]


In [15]:
submission.to_csv('catboost.csv', index=False)

In [30]:
def validate_submission(submission, test_df, num_users=1500, num_recs=50):
    """
    Validate submission dataframe for recommendation task.
    
    Parameters:
    -----------
    submission : pd.DataFrame
        Submission dataframe with columns: id, user_id, item_id, rank
    test_df : pd.DataFrame
        Test dataframe with user_id column
    num_users : int
        Expected number of unique users (default: 1500)
    num_recs : int
        Expected number of recommendations per user (default: 50)
    """
    # 1. Check total row count
    expected_rows = num_users * num_recs
    assert len(submission) == expected_rows, f"Expected {expected_rows} rows, got {len(submission)}"
    
    # 2. Check each user has exactly 50 recommendations
    user_counts = submission.groupby('user_id').size()
    assert (user_counts == num_recs).all(), \
        f"Not all users have {num_recs} recommendations. Counts: {user_counts[user_counts != num_recs]}"
    
    # 3. Check ranks are from 1 to 50 for each user
    for user_id, group in submission.groupby('user_id'):
        ranks = sorted(group['rank'].values)
        assert ranks == list(range(1, num_recs + 1)), \
            f"User {user_id} has invalid ranks: {ranks}"
    
    # 4. Check no duplicate item_ids per user
    duplicate_check = submission.groupby('user_id')['item_id'].apply(lambda x: len(x) != len(set(x)))
    assert not duplicate_check.any(), \
        f"Users with duplicate items: {duplicate_check[duplicate_check].index.tolist()}"
    
    # 5. Check all test users are present
    test_users = set(test_df['user_id'].unique())
    submission_users = set(submission['user_id'].unique())
    assert test_users == submission_users, \
        f"Missing users: {test_users - submission_users}, Extra users: {submission_users - test_users}"
    
    # 6. Check id column is sequential
    assert list(submission['id']) == list(range(len(submission))), \
        "ID column is not sequential"
    
    # 7. Check no null values
    assert not submission.isnull().any().any(), \
        f"Null values found in columns: {submission.columns[submission.isnull().any()].tolist()}"
    
    print("✅ All validation checks passed!")
    return True


validate_submission(submission, pd.read_csv('/kaggle/input/hear-me-personalized-music-recommender/test.csv'))

AssertionError: Expected 75000 rows, got 52400

In [23]:
submission.to_csv("base_coomatrix_4fact_30iters.csv",index=False)